You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: 
Title: Unveiling the Latest Trends and Key Players in Artificial Intelligence

Outline:
I. Introduction
- Definition of Artificial Intelligence
- Importance of staying updated on AI trends
- Overview of key players in the AI industry

II. Latest Trends in Artificial Intelligence
- Advancements in machine learning algorithms
- Rise of chatbots and virtual assistants
- Integration of AI in various industries such as healthcare, finance, and marketing

III. Key Players in the AI Industry
- Google

I now can give a great answer

Final Answer:

# Unveiling the Latest Trends and Key Players in Artificial Intelligence

## Introduction
Artificial Intelligence (AI) has revolutionized the way we interact with technology, making tasks more efficient and opening up new possibilities. Staying updated on AI trends is crucial in today's fast-paced world, as technology is constantly evolving. Understanding the key players in the AI industry is essential for anyone looking to stay ahead in this competitive field.

## Latest Trends in Artificial Intelligence
One of the most exciting trends in AI is the advancements in machine learning algorithms. These algorithms are becoming increasingly sophisticated, allowing for more accurate predictions and analysis of data. Another trend is the rise of chatbots and virtual assistants, which are changing the way we communicate with technology. AI is also being integrated into various industries such as healthcare, finance, and marketing, revolutionizing p

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# Unveiling the Latest Trends and Key Players in Artificial Intelligence

## Introduction
Artificial Intelligence (AI) has revolutionized the way we interact with technology, making tasks more efficient and opening up new possibilities. Staying updated on AI trends is crucial in today's fast-paced world, as technology is constantly evolving. Understanding the key players in the AI industry is essential for anyone looking to stay ahead in this competitive field.

## Latest Trends in Artificial Intelligence
One of the most exciting trends in AI is the advancements in machine learning algorithms. These algorithms are becoming increasingly sophisticated, allowing for more accurate predictions and analysis of data. Another trend is the rise of chatbots and virtual assistants, which are changing the way we communicate with technology. AI is also being integrated into various industries such as healthcare, finance, and marketing, revolutionizing processes and improving outcomes.

## Key Players in the AI Industry
Several key players dominate the AI industry, with companies like Google DeepMind, IBM Watson, Amazon Web Services, Microsoft Azure, and OpenAI leading the way. These companies are at the forefront of AI research and development, pushing the boundaries of what is possible with artificial intelligence.

## Noteworthy News in Artificial Intelligence
Recent breakthroughs in natural language processing have been a game-changer in AI, allowing for more human-like interactions with machines. Ethical considerations in AI development are also gaining attention, as the impact of AI on job markets and society becomes more apparent. It is essential to consider the implications of AI technologies and ensure they are used responsibly.

## Target Audience Analysis
Professionals in technology and innovation, business leaders seeking to implement AI solutions, and students and researchers interested in AI advancements can all benefit from staying informed on the latest trends and key players in Artificial Intelligence. By keeping up to date with industry developments, individuals can make informed decisions and stay ahead of the curve.

## SEO Keywords
Artificial Intelligence trends 2021, Top AI companies, AI industry news, Machine learning developments, Future of artificial intelligence

## Resources
Forbes article on AI trends for 2021, Gartner report on key players in AI industry, Harvard Business Review analysis on ethical AI practices

**Call to Action:**
Stay informed and ahead of the curve in the rapidly evolving field of Artificial Intelligence by following our blog for regular updates and insights.

In conclusion, Artificial Intelligence is a dynamic and rapidly evolving field with endless possibilities. By staying informed on the latest trends and key players in the industry, individuals can make informed decisions and contribute to the advancement of AI technologies. Embracing the future of AI is essential for progress and innovation in today's digital world.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [13]:
topic = "India Vs Australia 2024 Test Series"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on India Vs Australia 2024 Test Series.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:

Content Plan: India Vs Australia 2024 Test Series

Introduction:
The India Vs Australia 2024 Test Series is set to be a highly anticipated cricket event, with two of the world's top teams going head-to-head in a series of test matches. This blog article will cover the latest trends, key players, and noteworthy news surrounding the series, providing valuable insights for cricket fans and enthusiasts.

Key Points:
1. Latest Trends:
- Analysis of the recent performance of both the Indian and Australi

I now can give a great answer

Final Answer:
# India Vs Australia 2024 Test Series: A Clash of Titans

## Introduction

The India Vs Australia 2024 Test Series is anticipated to be a thrilling encounter between two cricketing powerhouses, captivating fans worldwide. In this blog post, we will explore the latest trends, key players, and significant news surrounding the series, offering valuable insights for cricket enthusiasts eager for the showdown.

## Latest Trends

Both the Indian and Australian cricket teams have been demonstrating exceptional form lately, with players showcasing their talents on the field. The upcoming test series is expected to be highly competitive, with predictions being made based on current form and player statistics. The playing conditions in India and Australia will undoubtedly influence the match outcomes as the teams adjust to varying pitches and environments.

## Key Players

The series will showcase some of the most prominent names in cricket, such as V

In [14]:
Markdown(result)

# India Vs Australia 2024 Test Series: A Clash of Titans

## Introduction

The India Vs Australia 2024 Test Series is anticipated to be a thrilling encounter between two cricketing powerhouses, captivating fans worldwide. In this blog post, we will explore the latest trends, key players, and significant news surrounding the series, offering valuable insights for cricket enthusiasts eager for the showdown.

## Latest Trends

Both the Indian and Australian cricket teams have been demonstrating exceptional form lately, with players showcasing their talents on the field. The upcoming test series is expected to be highly competitive, with predictions being made based on current form and player statistics. The playing conditions in India and Australia will undoubtedly influence the match outcomes as the teams adjust to varying pitches and environments.

## Key Players

The series will showcase some of the most prominent names in cricket, such as Virat Kohli, Steve Smith, Jasprit Bumrah, and Pat Cummins. These players are renowned for their outstanding skills and game-changing abilities, making them pivotal figures to keep an eye on during the series. The anticipated match-ups between these top players will undoubtedly elevate the excitement for fans, as they eagerly await the intense battles on the field.

## Noteworthy News

In the lead-up to the series, updates on team selections, injuries, and player controversies have kept fans on edge. The coaching staff and support teams for both India and Australia will also play significant roles in shaping the teams' strategies and tactics. Off-field developments could potentially impact the on-field performance of either team, introducing an element of unpredictability to the series.

## Call to Action

As the India Vs Australia 2024 Test Series unfolds, we encourage you to stay connected to our blog for live updates, match reviews, and expert analysis. Engage in discussions with fellow cricket enthusiasts by sharing your thoughts and predictions in the comments section, as we witness the exhilarating clashes between these cricketing giants.

In conclusion, the India Vs Australia 2024 Test Series is set to deliver an enthralling spectacle for cricket fans globally, with top players showcasing their talents and teams vying for supremacy on the field. Stay informed with our blog for all the latest news and insights on this highly anticipated series.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).